In [ ]:
import glob
import time
import pandas as pd
import numpy as np
import os
import platform
from pathlib import Path
import matplotlib.pyplot as plt
#astropy imports
from astropy.io import fits
from astropy.visualization import simple_norm
from astropy.stats import sigma_clipped_stats
from astropy.nddata import NDData
import ast
from astropy.nddata import StdDevUncertainty
from skimage.transform import resize
from astropy.table import Table, Column
from astropy.utils.data import get_pkg_data_filename
from astropy.wcs import WCS
from photutils.detection import find_peaks
from astropy.nddata.utils import Cutout2D
from photutils.background import MMMBackground
from photutils.psf import extract_stars
from photutils.psf.matching import resize_psf
from scipy.ndimage import shift

from photutils.centroids import centroid_com

# photutils imports
from photutils.detection import DAOStarFinder
from photutils.psf import PSFPhotometry, SourceGrouper, IntegratedGaussianPRF, extract_stars, EPSFBuilder, EPSFStar
# my modules
from cat_match import create_skycoord
#from psf_gen import return_filtered_table, build_psf_from_catalog, save_psf_to_fits
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

os_check = platform.platform(terse=True)[:5]
if os_check == 'macOS':
    preamble = '/Users/lordereinion/'
    root2 = f'{preamble}DropBox/Deconvolution/'
    root1 = f'{root2}Data/PHOTOMETRY/PHOTOM_CATS/'  
    root3 = f'{preamble}Dropbox/Deconvolution/'
else:
    preamble = '/home/six6ix6ix/'
    root2 = f'{preamble}Dropbox/Deconvolution/'
    root1 = f'{root2}Data/PHOTOMETRY/PHOTOM_CATS/'
    root3 = f'{preamble}Dropbox/Deconvolution/'




In [ ]:
psf_dir_native = f'{root2}PSFs_native/' 
psf_dir_rescaled = f'{root3}PSFs_rescaled/'

In [ ]:
def create_skycoord(ra, dec):
	# Create a SkyCoord object from RA and DEC arrays.
	return SkyCoord(ra=numpy.array(ra)*u.degree, dec=numpy.array(dec)*u.degree, frame='icrs')

def return_filtered_table(cluster, band, data_df, star_list):
    """
    Filters the input data frame by excluding rows where 'cPHOTID' is in the given star_list.

    Parameters:
    - cluster: Name of the cluster
    - band: The band of the image
    - data_df: DataFrame containing data for the cluster and band
    - star_list: A string or list of star IDs to filter

    Returns:
    - new_df: Filtered DataFrame
    """
    # If star_list is a string, try converting it to a list of integers
    if isinstance(star_list, str):
        try:
            list_of_strings = ast.literal_eval(star_list)
            if not isinstance(list_of_strings, list):
                raise ValueError("Input is not a list")
        except (ValueError, SyntaxError) as e:
            print(f"{cluster} {band} - Invalid star_list format: {star_list}")
            return data_df
    else:
        # If already a list, assign directly
        list_of_strings = star_list

    # Handle empty or invalid star list
    if not list_of_strings or list_of_strings == ["None"]:
        print(f"{cluster} {band} - No stars to filter")
        return data_df

    # Convert list of strings to integers, filtering out any non-integer values
    try:
        list_of_numbers = [int(num) for num in list_of_strings if str(num).isdigit()]
    except ValueError as e:
        print(f"{cluster} {band} - Error converting to integers: {e}")
        return data_df

    print(f"{cluster} {band} - Filtering out stars: {list_of_numbers}")
    
    # Filter the DataFrame
    new_df = data_df[~data_df['cPHOTID'].isin(list_of_numbers)]
    
    return new_df

def save_psf_to_fits(psf, output_filename):
	"""
	Save the PSF data to a FITS file.
	
	Parameters:
	- psf: The effective PSF object (epsf) returned by EPSFBuilder.
	- output_filename: The desired output path for the FITS file.
	"""
	# Create a new FITS HDU (Header/Data Unit) for the PSF data
	hdu = fits.PrimaryHDU(psf.data)
	
	# Create a HDU list to store in the FITS file
	hdul = fits.HDUList([hdu])
	
	# Write the HDU list to a new FITS file
	hdul.writeto(output_filename, overwrite=True)
	
	print(f"PSF saved to {output_filename}")


def build_psf_from_catalog_test(cluster, band, star_list, data_file, noise_file, catalog_file,
                           ra_col='ra_x', dec_col='dec_x', size=39, threshold=500.0):
    # Load the image data
    with fits.open(data_file) as hdul:
        image_data = hdul[0].data
        wcs = WCS(hdul[0].header)

    # Load the noise data
    with fits.open(noise_file) as hdu_n:
        noise_data = hdu_n[0].data

    # Create uncertainty using the noise data
    uncertainty = StdDevUncertainty(noise_data)
    
    # Load the catalog data
    catalog_unfiltered = pd.read_csv(catalog_file)
    catalog = return_filtered_table(cluster, band, catalog_unfiltered, star_list)
    
    # Extract RA and Dec from the catalog
    ra, dec = catalog[ra_col], catalog[dec_col]
    sky_coords = create_skycoord(list(ra), list(dec))
    
    # Convert the RA/Dec into pixel coordinates using WCS
    pixel_x, pixel_y = wcs.world_to_pixel(sky_coords)
    
    # Create a table for star positions in pixel coordinates
    stars_tbl = Table()
    stars_tbl['x'] = pixel_x
    stars_tbl['y'] = pixel_y

    # Perform sigma-clipping to remove background
    mean_val, median_val, std_val = sigma_clipped_stats(image_data, sigma=3.0)
    image_data -= median_val
    
    # Create NDData object for image data
    nddata = NDData(data=image_data)
    
    # Extract stars from the image using the pixel coordinates and a cutout size
    stars = extract_stars(nddata, stars_tbl, size=size)

    # Build the PSF using EPSFBuilder
    epsf_builder = EPSFBuilder(oversampling=1, maxiters=20, progress_bar=False, center_accuracy=1)
    epsf, fitted_stars = epsf_builder(stars)
    
    # Resize to match your scale
    psf = resize_psf(epsf.data, 0.2, 0.03)



    # Crop to 128x128
    n_to_crop = len(psf) - 128
    if n_to_crop % 2 == 0:
        x1 = int(n_to_crop / 2)
        x2 = int(len(psf) - n_to_crop / 2)
    else:
        x1 = int((n_to_crop + 1) / 2)
        x2 = int(len(psf) - (n_to_crop - 1) / 2)
    psf = psf[x1:x2, x1:x2]


    
    # === Centering using flux-weighted centroid ===
    centroid_y, centroid_x = centroid_com(psf)
    shift_y = 64.5 - centroid_y
    shift_x = 64.5 - centroid_x
    psf = shift(psf, shift=(shift_y, shift_x), order=3, mode='constant', cval=0.0)
    return psf